# The Hénon-Heiles Problem

### Nonlinear Dynamics, Symplectic Integration, and the Edge of Chaos

In the early 1960s, astronomers studying the motion of stars in galaxies faced a
puzzle. Classical mechanics says a system has as many conserved quantities
(**integrals of motion**) as degrees of freedom. For a star in an axisymmetric
galactic potential two integrals are obvious - total energy and angular momentum -
yet the observed stellar velocities were far more constrained than those two alone
could explain. It looked as if a mysterious **third integral** was keeping the
orbits on tighter leashes than energy and angular momentum permitted, but nobody
could write it down in closed form.

Michel Hénon (Observatoire de Nice) and Carl Heiles (then a graduate student, later
a Berkeley radio astronomer) attacked the question numerically in 1964. Rather than
model a full 3D galaxy, they stripped the problem to its mathematical skeleton: the
**simplest 2D nonlinear oscillator** that captures the essential physics. Their
question was whether the third integral truly exists in general, or whether it is
only an approximate feature of low-energy orbits that dissolves into chaos as the
energy rises.

**Why it matters.** This was one of the first clear numerical demonstrations that a
deterministic Hamiltonian system - no friction, no randomness, no forcing - can
produce motion that is effectively unpredictable. The work fed directly into what
became chaos theory, and it is a textbook illustration of the **KAM theorem**
(Kolmogorov-Arnold-Moser), which guarantees the survival of ordered, quasi-periodic
motion for most initial conditions at low energy while predicting its breakdown as
energy increases. The Hénon-Heiles system is still a standard benchmark for
numerical methods in Hamiltonian mechanics because its behavior is so rich: ordered
at low energy, mixed at intermediate energy, chaotic at high energy, all from one
compact set of equations.

This notebook reproduces the four classic diagnostic plots using the **Yoshida
4th-order symplectic integrator** (the same integrator used in `pendulums.ipynb` and
`planets.ipynb`), and follows the companion guide `henon_heiles.pdf`.

Reference: Hénon, M. & Heiles, C. (1964), *The Astronomical Journal*, **69**, 73.

---
## The mathematics

### The Hamiltonian

The system has two degrees of freedom: coordinates $q_1, q_2$ (two directions of
displacement in a 2D potential well) and their conjugate momenta $p_1, p_2$. With the
mass set to 1 the momenta equal the velocities. The Hamiltonian (the total energy) is

$$
H(q_1, q_2, p_1, p_2) =
\underbrace{\tfrac{1}{2}\left(p_1^2 + p_2^2\right)}_{\text{kinetic}}
+ \underbrace{\tfrac{1}{2}\left(q_1^2 + q_2^2\right)}_{\text{harmonic}}
+ \underbrace{\lambda\left(q_1^2 q_2 - \tfrac{1}{3}q_2^3\right)}_{\text{cubic coupling}}
$$

with $\lambda = 1$. The first two terms alone are two **uncoupled** simple harmonic
oscillators - a completely integrable system with forever-periodic motion. The cubic
term couples the two oscillators and is responsible for all the interesting nonlinear
behavior. It has the symmetry of an equilateral triangle (invariant under a 120°
rotation), which is why the accessible region of configuration space is triangular
(you will see this directly in Plot 2).

### Equations of motion

Hamilton's equations $\dot q_i = \partial H/\partial p_i$ and
$\dot p_i = -\partial H/\partial q_i$ give the positions

$$\dot q_1 = p_1, \qquad \dot q_2 = p_2,$$

and the momenta (the generalized forces)

$$
\dot p_1 = -q_1 - 2\lambda q_1 q_2, \qquad
\dot p_2 = -q_2 - \lambda\left(q_1^2 - q_2^2\right).
$$

The linear terms $-q_1, -q_2$ are the harmonic restoring forces; the nonlinear terms
couple the coordinates so a displacement in $q_1$ changes the force on $q_2$ and vice
versa. This coupling is what makes the system non-integrable in general. The four
first-order ODEs define a vector field in the 4D phase space $(q_1, q_2, p_1, p_2)$,
and each trajectory is confined to the 3D energy surface $H = E$.

In [ ]:
"""henon_heiles.ipynb"""

# Cell 01 - Imports and the Yoshida 4th-order coefficients

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

LAMBDA = 1.0  # cubic coupling constant (canonical Hénon-Heiles value)


def yoshida_coeffs() -> tuple[np.ndarray, np.ndarray]:
    """Position (c) and velocity (d) substep coefficients for Yoshida 4th order.

    From Yoshida (1990), "Construction of higher order symplectic integrators."
    The coefficients depend only on the cube root of 2 and are identical for any
    separable Hamiltonian H = T(p) + V(q) - the same coefficients used for the
    pendulum and the planetary orbits.
    """
    cbrt2 = 2.0 ** (1.0 / 3.0)
    w1 = 1.0 / (2.0 - cbrt2)
    w0 = -cbrt2 / (2.0 - cbrt2)
    c = np.array([w1 / 2.0, (w0 + w1) / 2.0, (w0 + w1) / 2.0, w1 / 2.0])
    d = np.array([w1, w0, w1])
    return c, d


c_coeff, d_coeff = yoshida_coeffs()
print(f"position coeffs c = {c_coeff}")
print(f"velocity coeffs d = {d_coeff}")
print(f"sum(c) = {c_coeff.sum():.6f}  sum(d) = {d_coeff.sum():.6f}  (both should be 1)")

In [ ]:
# Cell 02 - The Hamiltonian and the force law


def hamiltonian(q1, q2, p1, p2):
    """Total energy H = (p1^2 + p2^2)/2 + (q1^2 + q2^2)/2 + lambda(q1^2 q2 - q2^3/3)."""
    kinetic = 0.5 * (p1**2 + p2**2)
    potential = 0.5 * (q1**2 + q2**2) + LAMBDA * (q1**2 * q2 - q2**3 / 3.0)
    return kinetic + potential


def force(q1, q2):
    """Generalized forces (dp1/dt, dp2/dt) from -dH/dq.

    dp1/dt = -q1 - 2 lambda q1 q2
    dp2/dt = -q2 - lambda (q1^2 - q2^2)
    Works elementwise, so it accepts either scalars or arrays of trajectories.
    """
    f1 = -q1 - 2.0 * LAMBDA * q1 * q2
    f2 = -q2 - LAMBDA * (q1**2 - q2**2)
    return f1, f2


# Quick check: at the origin the potential and all forces vanish
f1_0, f2_0 = force(0.0, 0.0)
print(f"H(0,0,0,0)      = {hamiltonian(0.0, 0.0, 0.0, 0.0):.6f}  (expected 0)")
print(f"force at origin = ({f1_0:.6f}, {f2_0:.6f})  (expected 0, 0)")
print(f"H at q1=0.3,q2=0,p1=0.2,p2=0 = {hamiltonian(0.3, 0.0, 0.2, 0.0):.6f}")

---
## The critical energy and why the integrator matters

### The critical energy threshold

The cubic potential is **unbounded**: at large displacement the particle can escape
to infinity. The potential surface has three equivalent saddle points, all at energy
$E = 1/6 \approx 0.1667$. Below this threshold the particle is permanently confined;
above it, escape is possible. The character of the motion changes dramatically with
energy:

| Energy | Regime | Character |
|---|---|---|
| $E \ll 1/6$ (e.g. $E = 1/12$) | Subcritical | Quasi-periodic; smooth KAM tori; approximate third integral exists |
| $E \approx 1/8$ | Mixed | Islands of regular motion in a chaotic sea |
| $E \geq 1/6$ | Critical / supercritical | Predominantly chaotic; most tori destroyed; escape possible |

This notebook uses $E = 1/12 \approx 0.0833$, comfortably below the threshold, which
places the system squarely in the **ordered** regime. (Try $E = 1/6$ yourself and
watch the smooth curves in Plot 3 dissolve into a chaotic cloud.)

### Why a symplectic integrator

Hamiltonian systems preserve the phase-space volume element
$dq_1 \wedge dq_2 \wedge dp_1 \wedge dp_2$ for all time (Liouville's theorem), and
every trajectory lies exactly on the energy surface $H = \text{const}$ forever. A
standard solver such as RK4 does **not** respect this: over long runs it slowly leaks
or gains energy and the trajectory drifts off the true energy surface. Building a
useful Poincaré section here needs thousands of time units of integration, over which
that drift would scatter the crossing points spuriously - making it impossible to tell
genuine chaos from numerical error.

The **Yoshida 4th-order symplectic integrator** instead keeps the trajectory on a
nearby *shadow* energy surface that differs from the true $H$ by only $O(dt^4)$, and
this deviation does **not** accumulate with time. The energy error oscillates around a
fixed small level rather than drifting (Plot 4 shows exactly this). Each step composes
three weighted leapfrog (drift-kick-drift) substeps:

$$
\text{for } j = 0, 1, 2:\quad
\mathbf{q} \leftarrow \mathbf{q} + c_j\,\mathbf{p}\,dt,\quad
\mathbf{p} \leftarrow \mathbf{p} + d_j\,\mathbf{F}(\mathbf{q})\,dt
\qquad\text{then}\qquad
\mathbf{q} \leftarrow \mathbf{q} + c_3\,\mathbf{p}\,dt
$$

where $\mathbf{F}$ is evaluated at the already-updated position. The coefficients are
chosen so the $O(dt^2)$ and $O(dt^3)$ errors cancel exactly.

### Simulation setup

We integrate **six trajectories at once**, all at energy $E = 1/12$. Each shares
$q_2^0 = 0$ and $p_2^0 = 0$, with $p_1^0$ spread across the accessible range and
$q_1^0$ fixed by energy conservation, $q_1^0 = \sqrt{2E - (p_1^0)^2}$. Each runs for
$t_\text{final} = 5000$ s at $dt = 0.01$ (500,000 steps). Instead of looping over the
six trajectories separately, we carry all six as length-6 NumPy arrays and step them
together in one vectorized loop - the same integrator, just far faster.

In [ ]:
# Cell 03 - Build initial conditions, integrate all six trajectories, find crossings


def initial_conditions(energy, n_traj=6):
    """Six initial conditions on the energy surface H = energy.

    Set q2 = 0, p2 = 0 and spread p1 across [0.05, 0.92] * p1_max, then solve
    q1 = sqrt(2E - p1^2) from energy conservation. The endpoints are excluded to
    avoid degenerate fixed-point orbits.
    """
    p1_max = np.sqrt(2.0 * energy)
    p1_vals = np.linspace(0.05 * p1_max, 0.92 * p1_max, n_traj)
    q1_0 = np.sqrt(np.maximum(2.0 * energy - p1_vals**2, 0.0))
    q2_0 = np.zeros(n_traj)
    p2_0 = np.zeros(n_traj)
    return q1_0, q2_0, p1_vals, p2_0


def solve_yoshida4(q1_0, q2_0, p1_0, p2_0, t_final, dt):
    """Integrate all trajectories together with Yoshida's 4th-order method.

    Inputs are length-(n_traj) arrays of initial conditions. Returns the time
    array and four (n_steps, n_traj) arrays holding every trajectory's history.
    """
    c, d = yoshida_coeffs()
    n_steps = int(t_final / dt)
    n_traj = len(q1_0)
    t = np.arange(n_steps, dtype=np.float64) * dt

    q1 = np.zeros((n_steps, n_traj))
    q2 = np.zeros((n_steps, n_traj))
    p1 = np.zeros((n_steps, n_traj))
    p2 = np.zeros((n_steps, n_traj))
    q1[0], q2[0], p1[0], p2[0] = q1_0, q2_0, p1_0, p2_0

    r1, r2 = q1_0.copy(), q2_0.copy()
    s1, s2 = p1_0.copy(), p2_0.copy()
    for i in range(n_steps - 1):
        for j in range(3):
            r1 = r1 + c[j] * s1 * dt  # position drift
            r2 = r2 + c[j] * s2 * dt
            f1, f2 = force(r1, r2)
            s1 = s1 + d[j] * f1 * dt  # momentum kick at the updated position
            s2 = s2 + d[j] * f2 * dt
        r1 = r1 + c[3] * s1 * dt  # final position drift
        r2 = r2 + c[3] * s2 * dt
        q1[i + 1], q2[i + 1], p1[i + 1], p2[i + 1] = r1, r2, s1, s2

    return t, q1, q2, p1, p2


def poincare_section(q1_col, q2_col, p1_col, p2_col):
    """Return (q2, p2) where one trajectory crosses q1 = 0 moving with p1 > 0.

    Linear interpolation between the bracketing steps gives sub-step accuracy for
    the crossing without needing a finer integration grid.
    """
    below = q1_col[:-1] < 0.0
    above = q1_col[1:] >= 0.0
    rising = p1_col[:-1] > 0.0
    idx = np.where(below & above & rising)[0]
    frac = -q1_col[idx] / (q1_col[idx + 1] - q1_col[idx])
    sec_q2 = q2_col[idx] + frac * (q2_col[idx + 1] - q2_col[idx])
    sec_p2 = p2_col[idx] + frac * (p2_col[idx + 1] - p2_col[idx])
    return sec_q2, sec_p2


# --- Run the simulation ---
energy = 1 / 12  # subcritical: 1/8 for the mixed regime, 1/6 for chaos
t_final = 5000.0
dt = 0.01  # 500,000 steps; |dH| stays below ~1e-10 throughout
n_traj = 6

q1_0, q2_0, p1_0, p2_0 = initial_conditions(energy, n_traj)

print(
    f"Integrating {n_traj} trajectories at E = {energy:.5f} "
    f"for {t_final:.0f}s at dt = {dt} ({int(t_final / dt):,} steps each)..."
)
t, q1, q2, p1, p2 = solve_yoshida4(q1_0, q2_0, p1_0, p2_0, t_final, dt)
print("done.\n")

print(f"{'Traj':>4} {'q1_0':>8} {'p1_0':>8} {'crossings':>10}")
print("-" * 34)
total_crossings = 0
for k in range(n_traj):
    sq2, _ = poincare_section(q1[:, k], q2[:, k], p1[:, k], p2[:, k])
    total_crossings += len(sq2)
    print(f"{k + 1:>4} {q1_0[k]:>8.3f} {p1_0[k]:>8.3f} {len(sq2):>10}")
print(f"\nTotal Poincaré crossings across all trajectories: {total_crossings:,}")

---
## Why these four plots

The four plots are not arbitrary diagnostics. Each illuminates a different aspect of
the same physics, and together they tell a complete story: how the system behaves in
**time**, what region of **space** it explores, what its long-term **phase-space
structure** looks like, and whether the numerical method can be **trusted** enough to
believe any of the above.

### Plot 1 - Time series: $q_1(t)$ and $p_1(t)$

We plot two of the four coordinates - the position $q_1$ (crimson) and its conjugate
momentum $p_1 = \dot q_1$ (green) - over the first 200 seconds, for the 4th trajectory
(index 3), which starts near an equal split of energy between the two degrees of
freedom. The 20,000 steps in this window are subsampled at stride 5 for a smooth
curve.

The oscillations are clearly **not simple sinusoids**: the amplitude of both $q_1$ and
$p_1$ waxes and wanes on a slow timescale as energy sloshes back and forth between the
$q_1$ and $q_2$ degrees of freedom - **amplitude modulation**, or beating. This slow
envelope is the signature of two incommensurate frequencies (the fast oscillation
within a well and the slow exchange between wells) whose ratio is generally irrational,
so the motion never exactly repeats. The velocity $p_1$ leads the displacement $q_1$ by
roughly a quarter cycle, as in a simple oscillator, but the nonlinearity distorts that
relationship.

**Big-picture idea:** nonlinear quasi-periodic motion is a *third category* of behavior,
distinct from both clockwork periodicity and randomness. It has recognizable structure -
amplitude modulation, near-repetition, bounded oscillation - without ever exactly
repeating.

In [ ]:
# Cell 04 - Plot 1: time series q1(t) and p1(t) for trajectory 4 (first 200 s)

stride_ts = 5  # one point per 0.05 s
k = 3  # trajectory index 3 (the 4th)
show = (t <= 200.0) & (np.arange(len(t)) % stride_ts == 0)

plt.figure(figsize=(10, 5))
plt.plot(t[show], q1[show, k], color="crimson", lw=1.2, label=r"$q_1(t)$")
plt.plot(t[show], p1[show, k], color="forestgreen", lw=1.2, label=r"$p_1(t)$")
plt.title(r"Hénon-Heiles: Time Series  ($E = 1/12$, Yoshida 4th-Order)")
plt.xlabel("Time (s)")
plt.ylabel(r"$q_1$,  $p_1$")
plt.legend(framealpha=1.0, facecolor="white")
plt.grid(True)
plt.tight_layout()
plt.show()

### Plot 2 - Configuration space: $q_1$ vs. $q_2$

This projects the 4D trajectory onto the 2D plane of physical positions
$(q_1, q_2)$, stripped of momentum. All six trajectories are overlaid in distinct
colors; the 500,000-step trajectories are subsampled at stride 50 and drawn as tiny
dots to keep the plot from becoming an opaque mass.

Every trajectory is confined to the same bounded **triangular region**. This boundary
is not a numerical artifact - it is the **zero-velocity curve** of the potential,
$V(q_1, q_2) = E$, where all kinetic energy has become potential energy. The particle
cannot cross it because that would require negative kinetic energy. The triangular
shape with rounded corners is a direct consequence of the threefold symmetry of the
cubic term, and the region reaches further along $q_2$ than along $q_1$ because of the
asymmetry of the coupling.

**Big-picture idea:** energy conservation rigidly constrains which regions of space a
trajectory may visit, and the shape of that region is fixed entirely by the potential -
so it is the same for all six trajectories even though they start differently.

In [ ]:
# Cell 05 - Plot 2: configuration space q1 vs q2 for all six trajectories

stride_config = 50  # ~10,000 points per trajectory
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(8, 8))
for k in range(n_traj):
    label = f"Traj {k + 1}: $q_1^0={q1_0[k]:.3f}$, $p_1^0={p1_0[k]:.3f}$"
    plt.scatter(
        q1[::stride_config, k],
        q2[::stride_config, k],
        s=0.25,
        marker=".",
        color=cmap(k),
        label=label,
    )
plt.title(r"Hénon-Heiles: Configuration Space  ($E = 1/12$)")
plt.xlabel(r"$q_1$")
plt.ylabel(r"$q_2$")
plt.legend(
    loc="upper right",
    fontsize=7,
    framealpha=1.0,
    facecolor="white",
    markerscale=3,
    title="Initial conditions",
    title_fontsize=7,
)
plt.axis("equal")
plt.grid(True)
plt.tight_layout()
plt.show()

### Plot 3 - Poincaré section: $q_2$ vs. $p_2$ at $q_1 = 0,\ \dot q_1 > 0$

The Poincaré section is the most information-dense plot and the one most directly tied
to the theory of Hamiltonian chaos. Instead of plotting the trajectory continuously, we
slice the 4D phase space with the hyperplane $q_1 = 0$ and record only the $(q_2, p_2)$
coordinates at each crossing where $\dot q_1 = p_1 > 0$. This reduces a continuous 4D
trajectory to a discrete set of 2D points. No subsampling is applied - every crossing
is plotted.

At $E = 1/12$, the crossings for each trajectory lie on a **smooth, closed curve**
rather than scattering freely. These curves are the Poincaré slice of a **KAM torus** -
a 2D surface in the 4D phase space on which the trajectory is forever confined. The
curves nest concentrically and never intersect (two distinct trajectories at the same
energy can never pass through the same phase-space point). Near the center sits a
**fixed point**, the mark of a perfectly periodic orbit; the innermost curve belongs to
the trajectory with the most equal split of energy between $q_1$ and $q_2$.

**Big-picture idea:** quasi-periodic motion leaves a geometric fingerprint - smooth,
closed invariant curves. Chaotic motion would instead fill an area with a scattered
cloud of points. That visual contrast is one of the most powerful diagnostics in
Hamiltonian dynamics, and re-running at $E = 1/6$ turns these curves into exactly such a
cloud - a direct view of the KAM-to-chaos transition that is the central result of the
Hénon-Heiles paper.

In [ ]:
# Cell 06 - Plot 3: Poincaré section (q2, p2) at q1 = 0, dq1/dt > 0

plt.figure(figsize=(8, 8))
for k in range(n_traj):
    sq2, sp2 = poincare_section(q1[:, k], q2[:, k], p1[:, k], p2[:, k])
    if len(sq2) == 0:
        continue
    label = f"Traj {k + 1}: $q_1^0={q1_0[k]:.3f}$, $p_1^0={p1_0[k]:.3f}$"
    plt.scatter(sq2, sp2, s=4, color=cmap(k), label=label)
plt.title(r"Hénon-Heiles: Poincaré Section  $q_1=0,\ \dot{q}_1>0\ (E=1/12)$")
plt.xlabel(r"$q_2$")
plt.ylabel(r"$p_2$")
plt.legend(
    loc="upper right",
    fontsize=7,
    framealpha=1.0,
    facecolor="white",
    markerscale=2.5,
    title="Initial conditions",
    title_fontsize=7,
)
plt.grid(True)
plt.tight_layout()
plt.show()

### Plot 4 - Energy drift: $|H(t) - H_0|$

This semilog plot shows how far the numerically computed energy $H(t)$ strays from its
initial value $H_0 = E = 1/12$ over the full 5000-second run of trajectory 4. The
deviation is evaluated by substituting the computed $(q_1, q_2, p_1, p_2)$ back into the
Hamiltonian at each step, then subsampled at stride 250 for a clean envelope.

The energy error **does not grow with time**. It oscillates rapidly at a nearly constant
level, bounded above near $10^{-10}$ - more than nine orders of magnitude below the
energy itself ($\approx 0.083$). The spikes are not instability: they reflect the
integrator sampling a *shadow Hamiltonian* that oscillates closely around the true $H$,
with occasional deep dips where the two nearly coincide. A non-symplectic solver such as
RK4 would instead show the error growing **monotonically**, eventually large enough to
change the trajectory's qualitative behavior.

**Big-picture idea:** a symplectic integrator gives a different *kind* of guarantee than
a general-purpose solver. What matters is not merely that the instantaneous error is
small, but that it stays **bounded for all time** - which only a method that preserves
the geometric structure of Hamiltonian mechanics can promise. That is why symplectic
integrators are the gold standard for long-time simulation of conservative systems, from
celestial mechanics to molecular dynamics to the Hamiltonian quantum-simulation methods
this course builds toward.

In [ ]:
# Cell 07 - Plot 4: energy drift |H(t) - H0| for trajectory 4 (log scale)

stride_energy = 250  # 2,000 points across the full run
k = 3

e0 = hamiltonian(q1[0, k], q2[0, k], p1[0, k], p2[0, k])
delta_e = np.abs(hamiltonian(q1[:, k], q2[:, k], p1[:, k], p2[:, k]) - e0)
print(
    f"H0 = {e0:.6f}   max |H(t) - H0| = {np.max(delta_e):.3e}   "
    f"(> 9 orders of magnitude below H0)"
)

plt.figure(figsize=(10, 5))
plt.semilogy(t[::stride_energy], delta_e[::stride_energy], color="purple", lw=0.8)
plt.title(r"Hénon-Heiles: Energy Drift  $|\Delta H(t)|$  (Yoshida 4th-Order)")
plt.xlabel("Time (s)")
plt.ylabel(r"$|H(t) - H_0|$")
plt.grid(True, which="both")
plt.tight_layout()
plt.show()

---
## References

- Hénon, M. & Heiles, C. (1964). *The Applicability of the Third Integral of Motion:
  Some Numerical Experiments.* The Astronomical Journal, **69**, 73-79.
- Yoshida, H. (1990). *Construction of Higher Order Symplectic Integrators.* Physics
  Letters A, **150**(5-7), 262-268.
- Forest, E. & Ruth, R. D. (1990). *Fourth-Order Symplectic Integration.* Physica D,
  **43**(1), 105-117.
- Tabor, M. (1989). *Chaos and Integrability in Nonlinear Dynamics.* Wiley. (Chapter 3
  covers KAM theory and the Hénon-Heiles system in depth.)

See the companion guide `henon_heiles.pdf` for the full narrative background.